In [ ]:
!pip -q install transformers accelerate sentencepiece

### Basic Geography Tutor

In [ ]:
import re
import json
import torch
from collections import deque
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="auto"
)

model.eval()

print("Model loaded successfully.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully.


In [ ]:
def generate_text(prompt, max_new_tokens=500):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
class BasicGeographyTutor:

    SYSTEM_PROMPT = """
You are a Geography Tutor.

Teach geography accurately and clearly.

When answering:
- Explain the topic in simple language.
- Give a definition.
- Explain the main concept.
- Give an example.
- Give a short summary.

If the topic involves a process, explain its steps.

Answer the student's question directly.
"""

    def ask(self, question):

        prompt = f"""
{self.SYSTEM_PROMPT}

Student Question:
{question}

Answer:
"""

        return generate_text(prompt)

In [ ]:
basic_agent = BasicGeographyTutor()

question = "Explain the water cycle."

basic_answer = basic_agent.ask(question)

print(basic_answer)

The Water Cycle is a fundamental concept in meteorology that describes how water moves through Earth's atmosphere, oceans, and land. It consists of three main processes: evaporation, condensation, and precipitation.

1. **Evaporation**: This is the process where water from oceans, lakes, rivers, and other bodies of water turns into vapor (water vapor) as it evaporates from the surface of these bodies. Evaporation occurs when heat energy from the sun warms the water on the surface, causing it to turn into liquid or steam.

2. **Condensation**: Once the water vapor reaches the cooler air above, it cools down and changes back into liquid form. This process is called condensation. The temperature at which this transition happens varies depending on the atmospheric conditions.

3. **Precipitation**: When the condensed water droplets combine with tiny particles like dust or pollen, they fall back to the ground as rain, snow, sleet, or hail. Precipitation can occur in various forms such as ra

# Optimizations

### Prompt Optimization

In [ ]:
BASE_SYSTEM_PROMPT = """
You are an expert Geography Tutor.

Your goal is to teach accurately, clearly, and at the student's requested level.

Rules:
1. Stay focused on geography.
2. Explain concepts step-by-step.
3. Use examples when useful.
4. Do not invent facts.
5. Use clear headings and bullet points.
6. If a requested field is not applicable, write "Not applicable".
7. End with a concise summary.
"""

### Intent Classification

In [ ]:
def classify_intent(query):

    q = query.lower().strip()

    # 1. Definition
    if any(word in q for word in [
        "what is",
        "what are",
        "define",
        "definition",
        "meaning",
        "what do you mean by"
    ]):
        return "definition"

    # 2. Causes / Reasons
    if any(word in q for word in [
        "why",
        "cause",
        "causes",
        "reason",
        "reasons",
        "why does",
        "why do",
        "why is"
    ]):
        return "cause"

    # 3. Process / Formation
    if any(word in q for word in [
        "how does",
        "how do",
        "how is",
        "how are",
        "formation",
        "formed",
        "process",
        "develop",
        "development",
        "cycle"
    ]):
        return "process"

    # 4. Example
    if any(word in q for word in [
        "example",
        "examples",
        "give an example",
        "give examples",
        "illustrate",
        "instance"
    ]):
        return "example"

    # 5. Comparison
    if any(word in q for word in [
        "compare",
        "comparison",
        "difference",
        "differences",
        "different from",
        "versus",
        " vs ",
        "distinguish between"
    ]):
        return "comparison"

    # 6. Location
    if any(word in q for word in [
        "where",
        "location",
        "located",
        "located in",
        "found",
        "found in",
        "distribution",
        "distributed"
    ]):
        return "location"

    # 7. Effects / Impacts
    if any(word in q for word in [
        "effect",
        "effects",
        "impact",
        "impacts",
        "consequence",
        "consequences",
        "result",
        "results",
        "influence",
        "influences"
    ]):
        return "effects"

    # 8. Types / Classification
    if any(word in q for word in [
        "types",
        "type of",
        "kinds",
        "kind of",
        "classification",
        "classify",
        "categories",
        "category"
    ]):
        return "classification"

    # 9. Advantages / Disadvantages
    if any(word in q for word in [
        "advantages",
        "disadvantages",
        "benefits",
        "drawbacks",
        "pros",
        "cons",
        "importance",
        "significance"
    ]):
        return "advantages_disadvantages"

    # 10. General explanation
    if any(word in q for word in [
        "explain",
        "describe",
        "tell me about",
        "discuss",
        "elaborate"
    ]):
        return "explanation"

    return "general"

In [ ]:
queries = [
    "What is a monsoon?",
    "Why do monsoons occur?",
    "How are mountains formed?",
    "Give an example of a delta.",
    "Compare weather and climate.",
    "Where is the Sahara Desert located?",
    "What are the effects of deforestation?",
    "What are the types of rocks?",
    "What are the advantages of irrigation?",
    "Explain the water cycle."
]

for query in queries:
    print(f"{query} -> {classify_intent(query)}")

What is a monsoon? -> definition
Why do monsoons occur? -> cause
How are mountains formed? -> process
Give an example of a delta. -> example
Compare weather and climate. -> comparison
Where is the Sahara Desert located? -> location
What are the effects of deforestation? -> definition
What are the types of rocks? -> definition
What are the advantages of irrigation? -> definition
Explain the water cycle. -> process


### Dynamic Prompt Selection

In [ ]:
THEORY_PROMPT = """
Focus on geographical concepts and theory.

Structure:
## Topic
## Definition
## Explanation
## Key Points
## Example
## Summary
"""

# Define DEFINITION_PROMPT (it was referenced but not defined)
DEFINITION_PROMPT = THEORY_PROMPT

CAUSE_PROMPT = """
Focus on explaining the causes or reasons behind a geographical phenomenon.

Structure:
## Topic
## Definition
## Causes
Explain the major causes and why they occur.
## Example
Give a relevant geographical example.
## Summary
"""


PROCESS_PROMPT = """
Focus on explaining a geographical process or formation.

Structure:
## Topic
## Definition
## Process
Explain the process step-by-step in logical order.
## Example
Give a relevant geographical example.
## Summary
"""


EXAMPLE_PROMPT = """
Focus on providing clear geographical examples.

Structure:
## Topic
## Definition
## Examples
Give one or more real-world geographical examples.
Explain how each example relates to the topic.
## Key Points
## Summary
"""


COMPARISON_PROMPT = """
Focus on comparing two or more geographical concepts.

Structure:
## Topic
## Definitions
Briefly define each concept.

## Comparison

| Feature | Concept A | Concept B |
|---|---|---|
| Definition | | |
| Characteristics | | |
| Causes | | |
| Effects | | |
| Examples | | |

## Summary
Clearly state the main differences.
"""


LOCATION_PROMPT = """
Focus on the geographical location and distribution of a place or phenomenon.

Structure:
## Topic
## Definition
## Location
Explain where it is located.
## Distribution
Explain how and where it is distributed.
## Geographical Factors
Explain important physical or human factors affecting its location.
## Example
## Summary
"""


EFFECTS_PROMPT = """
Focus on the effects or impacts of a geographical phenomenon.

Structure:
## Topic
## Definition
## Causes
Briefly explain what causes the phenomenon.
## Effects
Explain the major physical, environmental, social, and economic effects where relevant.
## Example
Give a real-world geographical example.
## Summary
"""


CLASSIFICATION_PROMPT = """
Focus on the types or classification of a geographical concept.

Structure:
## Topic
## Definition
## Classification
List the major types or categories.
## Characteristics
Explain the important characteristics of each type.
## Examples
Give examples for each major type.
## Summary
"""


ADVANTAGES_DISADVANTAGES_PROMPT = """
Focus on the importance, benefits, limitations, or impacts of a geographical concept.

Structure:
## Topic
## Definition
## Advantages / Importance
Explain the major benefits or significance.
## Disadvantages / Limitations
Explain the major limitations or negative aspects.
## Example
Give a relevant geographical example.
## Summary
"""


EXPLANATION_PROMPT = """
Focus on giving a clear and complete explanation of the geographical topic.

Structure:
## Topic
## Definition
## Explanation
Explain the concept clearly from basic to deeper ideas.
## Key Points
## Example
## Summary
"""


GENERAL_PROMPT = """
Provide a clear and accurate geographical explanation.

Structure:
## Topic
## Definition
## Explanation
## Key Points
## Example
## Summary
"""


# Map the detected geography intent to the appropriate prompt.
INTENT_PROMPTS = {

    "definition": DEFINITION_PROMPT,

    "cause": CAUSE_PROMPT,

    "process": PROCESS_PROMPT,

    "example": EXAMPLE_PROMPT,

    "comparison": COMPARISON_PROMPT,

    "location": LOCATION_PROMPT,

    "effects": EFFECTS_PROMPT,

    "classification": CLASSIFICATION_PROMPT,

    "advantages_disadvantages":
        ADVANTAGES_DISADVANTAGES_PROMPT,

    "explanation": EXPLANATION_PROMPT,

    "general": GENERAL_PROMPT
}

### Clarification Agent

In [ ]:
AMBIGUOUS_TOPICS = {
    "climate": [
        "climate definition",
        "climate classification",
        "climate change",
        "climate zones"
    ],
    "weather": [
        "weather definition",
        "weather vs climate",
        "weather forecasting",
        "weather elements"
    ],
    "soil": [
        "soil types",
        "soil formation",
        "soil profile",
        "soil erosion"
    ],
    "rivers": [
        "river formation",
        "river erosion",
        "river deposition",
        "river landforms"
    ],
    "mountains": [
        "mountain formation",
        "types of mountains",
        "mountain ranges",
        "mountain landforms"
    ]
}


def detect_ambiguity(query):

    q = query.lower().strip()

    # Very short questions are often ambiguous
    words = q.split()

    for topic, options in AMBIGUOUS_TOPICS.items():

        if topic in q and len(words) <= 4:

            return {
                "ambiguous": True,
                "topic": topic,
                "options": options
            }

    return {
        "ambiguous": False
    }

In [ ]:
def clarification_message(ambiguity):

    topic = ambiguity["topic"]
    options = ambiguity["options"]

    return (
        f"Could you clarify what you mean by **{topic}**?\n\n"
        + "\n".join(
            f"{i+1}. {option}"
            for i, option in enumerate(options)
        )
    )

### Adaptive Tutoring

In [ ]:
LEVEL_PROMPTS = {

    "beginner": """
Student level: Beginner.

Use simple language.
Explain basic terminology.
Use everyday examples.
Avoid unnecessary technical details.
""",

    "intermediate": """
Student level: Intermediate.

Assume basic geography knowledge.
Explain mechanisms and relationships.
Include useful technical terminology.
""",

    "advanced": """
Student level: Advanced.

Use precise geographical terminology.
Discuss mechanisms, exceptions, and deeper relationships.
Assume strong foundational knowledge.
"""
}


def normalize_level(level):

    level = level.lower().strip()

    if level not in LEVEL_PROMPTS:
        return "beginner"

    return level

### Context Awareness

In [ ]:
class ConversationMemory:

    def __init__(self, max_items=2):
        self.history = deque(maxlen=max_items)

    def add(self, question, answer):

        self.history.append({
            "question": question,
            "answer": answer
        })

    def get_context(self):

        if not self.history:
            return "No previous conversation."

        context = []

        for item in self.history:

            context.append(
                f"Student: {item['question']}\n"
                f"Tutor: {item['answer']}"
            )

        return "\n\n".join(context)

    def clear(self):
        self.history.clear()

### Response Validation

In [ ]:
def validate_response(response, required_sections):
    issues = []

    for section in required_sections:
        if section not in response:
            issues.append(f"Missing section: {section.replace('## ', '')}")

    return {
        "valid": len(issues) == 0,
        "missing": issues,
        "issues": issues
    }


### Self-Reflection

In [ ]:
REFLECTION_PROMPT = """
You are reviewing a Geography Tutor's answer.

Check the answer for:

1. Missing information
2. Geographical factual errors
3. Missing required sections
4. Poor formatting
5. Incomplete explanation
6. Repetition

{validation_feedback}

Your response MUST be ONLY a JSON object, formatted as follows:

```json
{{
    "is_good": true or false,
    "issues": [
        "issue 1",
        "issue 2"
    ],
    "improvement_instructions": "specific instructions"
}}
```

Do not include any other text, comments, or explanations outside of the JSON markdown block.

Student question:
{question}

Tutor answer:
{answer}
"""

In [ ]:
def extract_json(text):
    # Try to extract from a markdown code block first
    code_block_match = re.search(r"```json\s*(\{.*\})\s*```", text, re.DOTALL)
    if code_block_match:
        json_str = code_block_match.group(1)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to general JSON extraction

    # Fallback: general JSON extraction
    match = re.search(
        r"\{.*\}",
        text,
        re.DOTALL
    )

    if not match:
        return None

    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

In [ ]:
def self_reflect(question, answer, validation_result):

    # Start with issues from programmatic validation
    issues = validation_result.get("issues", [])
    is_good = validation_result.get("valid", True) # Assume good unless validation explicitly says otherwise

    # Construct feedback for the LLM based on programmatic validation
    validation_feedback = ""
    if not is_good:
        validation_feedback = f"The following sections were identified as missing by an automated check: {', '.join(issues)}. "
        validation_feedback += "Please ensure these are present and address any other issues."

    prompt = REFLECTION_PROMPT.format(
        question=question,
        answer=answer,
        validation_feedback=validation_feedback # New format argument
    )

    raw = generate_text(
        prompt,
        max_new_tokens=300
    )

    result_from_llm = extract_json(raw)

    if result_from_llm is None:
        # Fallback if LLM's JSON is bad
        return {
            "is_good": False,
            "issues": issues + ["Reflection LLM JSON parsing failed"],
            "improvement_instructions": "Review and improve the answer, and ensure the self-reflection response is valid JSON."
        }

    # Combine issues from programmatic validation and LLM's reflection
    # Use set to avoid duplicates, then convert back to list
    combined_issues = list(set(issues + result_from_llm.get("issues", [])))

    # Determine final is_good status
    # Must be good from both programmatic validation AND LLM's reflection
    final_is_good = is_good and result_from_llm.get("is_good", False)

    return {
        "is_good": final_is_good,
        "issues": combined_issues,
        "improvement_instructions": result_from_llm.get("improvement_instructions", "")
    }

### Retry Strategy

In [ ]:
def generate_with_retry(
    tutor_instance, # New argument to access build_prompt
    prompt,
    question,
    intent="general",
    max_retries=2
):

    last_response = ""

    # Define intent-specific required sections for validation
    intent_validation_sections = {
        "definition": ["## Topic", "## Definition", "## Explanation", "## Key Points", "## Example", "## Summary"],
        "cause": ["## Topic", "## Definition", "## Causes", "## Example", "## Summary"],
        "process": ["## Topic", "## Definition", "## Process", "## Example", "## Summary"],
        "example": ["## Topic", "## Definition", "## Examples", "## Key Points", "## Summary"],
        "comparison": ["## Topic", "## Definitions", "## Comparison", "## Example", "## Summary"],
        "location": ["## Topic", "## Definition", "## Location", "## Distribution", "## Geographical Factors", "## Example", "## Summary"],
        "effects": ["## Topic", "## Definition", "## Causes", "## Effects", "## Example", "## Summary"],
        "classification": ["## Topic", "## Definition", "## Classification", "## Characteristics", "## Examples", "## Summary"],
        "advantages_disadvantages": ["## Topic", "## Definition", "## Advantages / Importance", "## Disadvantages / Limitations", "## Example", "## Summary"],
        "explanation": ["## Topic", "## Definition", "## Explanation", "## Key Points", "## Example", "## Summary"],
        "general": ["## Topic", "## Definition", "## Explanation", "## Key Points", "## Example", "## Summary"]
    }
    # Get the specific required sections for the current intent
    current_required_sections = intent_validation_sections.get(intent, intent_validation_sections["general"])

    for attempt in range(max_retries + 1):

        response = generate_text(
            prompt,
            max_new_tokens=700
        )

        last_response = response

        # Validate according to geography intent
        validation = validate_response(
            response,
            required_sections=current_required_sections
        )

        # Response is complete
        if validation["valid"]:
            return response

        # -----------------------------
        # Retry with stricter prompt
        # -----------------------------

        missing_sections = validation["missing"]

        retry_feedback = f"""
Your previous answer was missing the following sections: {", ".join(missing_sections)}.
Please regenerate the answer ensuring all required sections are present.
"""

        # Rebuild the prompt using the tutor instance's build_prompt method
        # This prevents duplication of the original question and instructions.
        prompt = tutor_instance.build_prompt(
            question,
            intent,
            tutor_instance.memory.get_context(), # Get fresh context for the new prompt
            reflection=retry_feedback # Pass the retry feedback as reflection
        )

    # Return the last generated response if all retries fail
    return last_response

### Response Refinement

In [ ]:
def refine_response(response):

    # Normalize headings
    replacements = {
        "### Topic": "## Topic",
        "### Definition": "## Definition",
        "### Explanation": "## Explanation",
        "### Process": "## Process",
        "### Example": "## Example",
        "### Summary": "## Summary"
    }

    for old, new in replacements.items():
        response = response.replace(old, new)

    # Remove excessive blank lines
    response = re.sub(
        r"\n{3,}",
        "\n\n",
        response
    )

    # Remove repeated consecutive sentences
    lines = response.splitlines()

    cleaned = []
    previous = None

    for line in lines:

        normalized = line.strip().lower()

        if normalized and normalized == previous:
            continue

        cleaned.append(line)

        if normalized:
            previous = normalized

    response = "\n".join(cleaned).strip()

    return response

### Putting all these optimizations together

In [ ]:
class OptimizedGeographyTutor:

    def __init__(self, level="beginner"):

        self.level = normalize_level(level)

        self.memory = ConversationMemory(
            max_items=2
        )

    def set_level(self, level):

        self.level = normalize_level(level)

    def _get_required_sections_format_string(self, intent):
        # This mirrors the intent_requirements in generate_with_retry
        intent_formats = {
            "definition": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Explanation
Explain the core concept.

## Key Points
List key aspects.

## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
""",
            "cause": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Causes
Explain the major causes.

## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
""",
            "process": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Process
Explain the mechanism, process, or major steps.

## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
""",
            "example": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Examples
Give one or more real-world geographical examples.

## Key Points
List key aspects.

## Summary
Give 2-3 concise points.
""",
            "comparison": """
## Topic
Identify the geographical topic.

## Definitions
Briefly define each concept.

## Comparison
Provide a comparison, e.g., using a table.

## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
""",
            "location": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Location
Explain where it is located.

## Distribution
Explain how and where it is distributed.

## Geographical Factors
Explain important factors affecting its location.
## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
""",
            "effects": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Causes
Briefly explain what causes the phenomenon.

## Effects
Explain the major effects.

## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
""",
            "classification": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Classification
List the major types or categories.

## Characteristics
Explain the important characteristics of each type.

## Examples
Give examples for each major type.

## Summary
Give 2-3 concise points.
""",
            "advantages_disadvantages": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Advantages / Importance
Explain the major benefits or significance.

## Disadvantages / Limitations
Explain the major limitations or negative aspects.
## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
""",
            "explanation": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Explanation
Explain the concept clearly from basic to deeper ideas.

## Key Points
List key aspects.

## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
""",
            "general": """
## Topic
Identify the geographical topic.

## Definition
Give a clear definition.

## Explanation
Explain the core concept.

## Key Points
List key aspects.

## Example
Give a concrete geographical example.

## Summary
Give 2-3 concise points.
"""
        }
        return intent_formats.get(intent, intent_formats["general"])

    def build_prompt(
        self,
        question,
        intent,
        context,
        reflection=None
    ):

        task_prompt = INTENT_PROMPTS.get(
            intent,
            THEORY_PROMPT
        )

        reflection_instruction = ""

        if reflection:

            reflection_instruction = f"""
Previous reviewer feedback:

{reflection}

Fix all identified problems.
"""

        prompt = f"""
{BASE_SYSTEM_PROMPT}

{LEVEL_PROMPTS[self.level]}

Learning objective:
{intent}

Task-specific instructions:
{task_prompt}

Conversation context:
{context}

{reflection_instruction}

Student question:
{question}

Return only the tutoring answer.
"""

        return prompt

    def ask(self, question):

        # --------------------------------
        # Optimization 2
        # Intent classification
        # --------------------------------

        intent = classify_intent(question)

        # --------------------------------
        # Optimization 4
        # Clarification
        # --------------------------------

        ambiguity = detect_ambiguity(question)

        if ambiguity["ambiguous"]:

            return clarification_message(
                ambiguity
            )

        # --------------------------------
        # Optimization 9
        # Context awareness
        # --------------------------------

        context = self.memory.get_context()

        # --------------------------------
        # Optimization 3
        # Dynamic prompt
        # --------------------------------

        prompt = self.build_prompt(
            question,
            intent,
            context
        )

        # --------------------------------
        # Optimization 7
        # Retry
        # --------------------------------

        answer = generate_with_retry(
            self, # Pass self (tutor instance) to generate_with_retry
            prompt,
            question,
            intent=intent,
            max_retries=2
        )

        # --------------------------------
        # Optimization 5
        # Validation
        # --------------------------------

        # The initial validation call in generate_with_retry now correctly uses intent-specific sections.
        # This call was removed because it was redundant and might use incorrect default sections.

        # --------------------------------
        # Optimization 6
        # Self-reflection
        # --------------------------------

        # The validation in generate_with_retry also feeds into self-reflection if a retry happens.
        # The original self-reflection call here needs to be after an initial validation from generate_with_retry.
        # Let's ensure a robust validation is applied to the final answer from generate_with_retry.

        # Re-evaluate validation for the final answer after retries (if any)
        # This will use the same logic as the initial validation in generate_with_retry
        intent_validation_sections = {
            "definition": ["## Topic", "## Definition", "## Explanation", "## Key Points", "## Example", "## Summary"],
            "cause": ["## Topic", "## Definition", "## Causes", "## Example", "## Summary"],
            "process": ["## Topic", "## Definition", "## Process", "## Example", "## Summary"],
            "example": ["## Topic", "## Definition", "## Examples", "## Key Points", "## Summary"],
            "comparison": ["## Topic", "## Definitions", "## Comparison", "## Example", "## Summary"],
            "location": ["## Topic", "## Definition", "## Location", "## Distribution", "## Geographical Factors", "## Example", "## Summary"],
            "effects": ["## Topic", "## Definition", "## Causes", "## Effects", "## Example", "## Summary"],
            "classification": ["## Topic", "## Definition", "## Classification", "## Characteristics", "## Examples", "## Summary"],
            "advantages_disadvantages": ["## Topic", "## Definition", "## Advantages / Importance", "## Disadvantages / Limitations", "## Example", "## Summary"],
            "explanation": ["## Topic", "## Definition", "## Explanation", "## Key Points", "## Example", "## Summary"],
            "general": ["## Topic", "## Definition", "## Explanation", "## Key Points", "## Example", "## Summary"]
        }
        current_required_sections_for_final_validation = intent_validation_sections.get(intent, intent_validation_sections["general"])

        validation = validate_response(
            answer,
            required_sections=current_required_sections_for_final_validation
        )

        reflection = self_reflect(
            question,
            answer,
            validation # Pass validation result to self_reflect
        )

        # If the answer is still not good after self-reflection (or if initial validation failed here),
        # we can attempt one more retry with specific improvement instructions.
        if not validation["valid"] or not reflection.get("is_good", False):
            improvement = reflection.get(
                "improvement_instructions",
                "Improve the answer by addressing missing sections or issues." # More specific instruction
            )

            improved_prompt = self.build_prompt(
                question,
                intent,
                context,
                reflection=improvement
            )

            answer = generate_with_retry(
                self, # Pass self for the retry
                improved_prompt,
                question,
                intent=intent,
                max_retries=1 # Only one more retry to avoid infinite loops
            )

        # --------------------------------
        # Optimization 10
        # Refinement
        # --------------------------------

        answer = refine_response(
            answer
        )

        # --------------------------------
        # Store context
        # --------------------------------

        self.memory.add(
            question,
            answer
        )

        return answer

In [ ]:
optimized_agent = OptimizedGeographyTutor(
    level="beginner"
)

question = "Explain the water cycle."

optimized_answer = optimized_agent.ask(
    question
)

print(optimized_answer)

## Topic
The Water Cycle

## Definition
The water cycle is a continuous process that involves the movement of water between the Earth's oceans, atmosphere, and land surface. It includes several key steps:

1. **Evaporation**: Water from oceans, lakes, rivers, and other bodies of water turns into vapor through heat absorption.
2. **Condensation**: The vapor rises due to gravity, forming clouds.
3. **Precipitation**: When the cloud collects enough moisture, it releases water as precipitation (rain, snow, sleet, or hail).
4. **Collection**: Precipitated water falls back to the ground as rain, snow, sleet, or hail.
5. **Runoff**: Some of this runoff flows over the land, replenishing groundwater supplies.
6. **Collection**: The remaining water infiltrates the soil, replenishes groundwater supplies, and eventually returns to the oceans via evaporation.

## Process
1. **Evaporation**: Water from oceans, lakes, rivers, and other bodies of water turns into vapor through heat absorption.
2. **Co

### Compare Basic vs Optimized

### Test Intent Classification

In [ ]:
test_queries = [
    "What is a delta?",
    "Explain how a delta forms.",
    "Compare weather and climate.",
    "Give an example of a volcanic island.",
    "Write Python code to calculate distance between two coordinates.",
    "Debug this geography program."
]

for query in test_queries:

    intent = classify_intent(query)

    print(f"Query  : {query}")
    print(f"Intent : {intent}")
    print("-" * 60)

Query  : What is a delta?
Intent : definition
------------------------------------------------------------
Query  : Explain how a delta forms.
Intent : explanation
------------------------------------------------------------
Query  : Compare weather and climate.
Intent : comparison
------------------------------------------------------------
Query  : Give an example of a volcanic island.
Intent : example
------------------------------------------------------------
Query  : Write Python code to calculate distance between two coordinates.
Intent : general
------------------------------------------------------------
Query  : Debug this geography program.
Intent : general
------------------------------------------------------------


### Test Clarification Agent

In [ ]:
optimized_agent.set_level("beginner")
print(optimized_agent.ask("Explain climate."))

Could you clarify what you mean by **climate**?

1. climate definition
2. climate classification
3. climate change
4. climate zones


### Test Adaptive Tutoring

In [ ]:
optimized_agent.set_level("beginner")

# For 'Compare the water cycle and the carbon cycle.', the intent will be 'comparison'.
# We need the required sections for comparison here to properly validate if needed.
required_sections_comparison = [
    "## Topic",
    "## Definitions",
    "## Comparison",
    "## Example",
    "## Summary"
]

# Get the answer from the tutor
answer_comparison = optimized_agent.ask("Compare the water cycle and the carbon cycle.")
print("QUESTION: Compare the water cycle and the carbon cycle.")
print(answer_comparison)

# Validate the answer if needed for a final check outside optimized_agent.ask
# validation_comparison = validate_response(
#     answer_comparison,
#     required_sections=required_sections_comparison
# )
# print("\nValidation for comparison:")
# print(json.dumps(validation_comparison, indent=2))

QUESTION: Compare the water cycle and the carbon cycle.
## Topic
Water Cycle vs Carbon Cycle

## Definition
- **Water Cycle** describes the movement of water between the Earth's oceans, atmosphere, and land surface.
  - Evaporation: Water from oceans, lakes, rivers, and other bodies of water turns into vapor through heat absorption.
  - Condensation: The vapor rises due to gravity, forming clouds.
  - Precipitation: When the cloud collects enough moisture, it releases water as precipitation (rain, snow, sleet, or hail).
  - Collection: Precipitated water falls back to the ground as rain, snow, sleet, or hail.
  - Runoff: Some of this runoff flows over the land, replenishing groundwater supplies.
  - Collection: The remaining water infiltrates the soil, replenishes groundwater supplies, and eventually returns to the oceans via evaporation.

- **Carbon Cycle** describes the movement of carbon dioxide (CO₂) between the atmosphere and the ocean.
  - Absorption: Plants absorb CO₂ from the a

In [ ]:
optimized_agent.set_level("intermediate")
print(optimized_agent.ask("Explain plate tectonics."))

Plate tectonics is a theory explaining the distribution of the Earth's lithosphere (the outer layer of the planet) into multiple plates that move relative to each other. These movements cause earthquakes, volcanic activity, and the formation of mountains and islands. The theory suggests that the Earth's crust is divided into several large plates that float on the mantle beneath them. Over time, these plates collide and push against each other, causing them to move apart or converge towards one another. This movement creates stress on the plates, which can lead to earthquakes, tsunamis, and other geological phenomena. Plate tectonics is crucial for understanding the Earth's geology and its ability to support life on our planet.


In [ ]:
optimized_agent.set_level("advanced")
print(optimized_agent.ask("Explain plate tectonics."))

Plate tectonics is a theory explaining the distribution of the Earth's lithosphere (the outer layer of the planet) into multiple plates that move relative to each other. These movements cause earthquakes, volcanic activity, and the formation of mountains and islands. The theory suggests that the Earth's crust is divided into several large plates that float on the mantle beneath them. Over time, these plates collide and push against each other, causing them to move apart or converge towards one another. This movement creates stress on the plates, which can lead to earthquakes, tsunamis, and other geological phenomena. Plate tectonics is crucial for understanding the Earth's geology and its ability to support life on our planet.


### Test Context Awareness

In [ ]:
optimized_agent = OptimizedGeographyTutor(level="intermediate")
answer1 = optimized_agent.ask("Explain the water cycle.")
print("QUESTION 1: Explain the water cycle.")
print(answer1)
print("\n" + "=" * 80 + "\n")
answer2 = optimized_agent.ask("Compare it with the carbon cycle.")
print("QUESTION 2: Compare the water cycle and the carbon cycle.")
print(answer2)

QUESTION 1: Explain the water cycle.
## Topic
The Water Cycle

## Definition
The water cycle is a continuous process that involves the movement of water between the Earth's oceans, atmosphere, and land surface. It includes several key processes:

1. **Evaporation**: Water from oceans, lakes, rivers, and other bodies of water turns into vapor through heat absorption.
2. **Condensation**: The vapor rises due to its lower temperature than the surrounding air, forming clouds.
3. **Precipitation**: When the cloud collects enough moisture, it releases water as precipitation (rain, snow, sleet, or hail).
4. **Collection**: Precipitated water falls back to the ground as rain, snow, sleet, or hail.
5. **Runoff**: This water flows over the land and eventually returns to bodies of water like rivers and lakes.

## Process
1. **Evaporation**: Water from oceans, lakes, rivers, and other bodies of water turns into vapor through heat absorption.
2. **Condensation**: The vapor rises due to its lower te

In [ ]:
bad_answer = """
## Topic
Water cycle

## Definition
Movement of water through Earth's systems.

## Example
Rainfall.
"""

# Define the required sections for this test case
required_sections_for_test = [
    "## Topic",
    "## Definition",
    "## Process",
    "## Example",
    "## Summary"
]

result = validate_response(
    bad_answer,
    required_sections=required_sections_for_test
)

print(json.dumps(
    result,
    indent=2
))

{
  "valid": false,
  "missing": [
    "Missing section: Process",
    "Missing section: Summary"
  ],
  "issues": [
    "Missing section: Process",
    "Missing section: Summary"
  ]
}


### Demonstrate Self-Reflection

In [ ]:
sample_question = "Explain ocean currents."

sample_answer = """
## Topic
Ocean currents

## Definition
Ocean currents are large-scale movements of seawater.

## Process
Ocean currents are influenced by wind, Earth's rotation,
temperature and salinity.

## Example
The Gulf Stream.

## Summary
Ocean currents redistribute heat around the planet.
"""

# Define required sections for 'explanation' intent
required_sections_explanation = [
    "## Topic",
    "## Definition",
    "## Explanation",
    "## Key Points",
    "## Example",
    "## Summary"
]

# Validate the sample_answer to get a validation_result
validation_for_good_answer = validate_response(
    sample_answer,
    required_sections=required_sections_explanation
)

reflection = self_reflect(
    sample_question,
    sample_answer,
    validation_for_good_answer # Pass the validation result
)

print(json.dumps(
    reflection,
    indent=2
))

{
  "is_good": false,
  "issues": [
    "Missing information",
    "Missing section: Key Points",
    "Geographical factual errors",
    "Missing section: Explanation"
  ],
  "improvement_instructions": "Add more examples to illustrate how ocean currents work."
}


In [ ]:
sample_question_bad = "Explain the water cycle."

sample_answer_bad = """
## Topic
Water cycle

## Definition
Movement of water through Earth's systems.

## Example
Rainfall.
"""

# Define required sections for 'process' intent (as water cycle is a process)
required_sections_process = [
    "## Topic",
    "## Definition",
    "## Process",
    "## Example",
    "## Summary"
]

# Validate the sample_answer_bad to get a validation_result
validation_for_bad_answer = validate_response(
    sample_answer_bad,
    required_sections=required_sections_process
)

reflection_bad = self_reflect(
    sample_question_bad,
    sample_answer_bad,
    validation_for_bad_answer # Pass the validation result
)

print(json.dumps(
    reflection_bad,
    indent=2
))

{
  "is_good": false,
  "issues": [
    "Missing section: Summary",
    "Geographical factual errors",
    "Missing information",
    "Missing section: Process"
  ],
  "improvement_instructions": "Add more examples to illustrate the water cycle."
}


## Demonstrating All Optimizations with the OptimizedGeographyTutor

In [ ]:
# Initialize the OptimizedGeographyTutor
final_tutor = OptimizedGeographyTutor(level="beginner")

print("--- Optimized Tutor Demonstration ---")

# 1. Demonstrate Intent Classification (Definition)
print("\nQuestion (Definition - Beginner): What is a delta?")
response = final_tutor.ask("What is a delta?")
print(response)

# 2. Demonstrate Adaptive Tutoring (Intermediate Level)
final_tutor.set_level("intermediate")
print("\nQuestion (Process - Intermediate): How are volcanoes formed?")
response = final_tutor.ask("How are volcanoes formed?")
print(response)

# 3. Demonstrate Clarification Agent
print("\nQuestion (Ambiguous): Explain climate.")
response = final_tutor.ask("Explain climate.")
print(response)

# 4. Demonstrate Context Awareness (follow-up question after asking about water cycle)
# Clear memory first to show fresh context awareness
final_tutor.memory.clear()
print("\nQuestion 1 (Water Cycle): Explain the water cycle.")
response1 = final_tutor.ask("Explain the water cycle.")
print(response1)

print("\nQuestion 2 (Context-aware Comparison): Compare it with the carbon cycle.")
response2 = final_tutor.ask("Compare it with the carbon cycle.")
print(response2)

# 5. Demonstrate Comparison Intent (explicitly)
print("\nQuestion (Comparison): Compare glaciers and icebergs.")
response = final_tutor.ask("Compare glaciers and icebergs.")
print(response)

# 6. Demonstrate Adaptive Tutoring (Advanced Level)
final_tutor.set_level("advanced")
print("\nQuestion (Effects - Advanced): Discuss the long-term effects of desertification.")
response = final_tutor.ask("Discuss the long-term effects of desertification.")
print(response)

--- Optimized Tutor Demonstration ---

Question (Definition - Beginner): What is a delta?
A delta is a small area of land that forms as a result of a river or stream cutting through a mountain range. It is characterized by its steep sides, narrow valleys, and often irregular shapes. Delta regions can be found in many parts of the world, including Asia, Europe, and North America. They play a crucial role in shaping landscapes and ecosystems, influencing water flow patterns and climate zones. Understanding deltas helps geographers and environmental scientists analyze how rivers and streams affect the landscape over time.

Question (Process - Intermediate): How are volcanoes formed?
Volcanic activity involves the sudden release of heat from within Earth's crust due to tectonic movements. This heat causes magma to rise and form new rock formations, which we call lava. The movement of magma into the Earth's crust creates fractures called volcanic vents, allowing it to escape back out into t